# Analyse exploratoire des données

## Objectif

L'objectif de ce notebook est d'explorer le jeu de données sur les films, de comprendre sa structure, d'identifier les valeurs manquantes et de préparer les données pour le tableau de bord Streamlit.

## Chargement des données
   - importer les  libraries
   - aperçu du jeu de données (dataset preview)

In [ ]:
# Load libraries and files
import pandas as pd

import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data_loader import load_data
from src.preprocessing import preprocess

import plotly.express as px

# Suppress warnings 
import warnings
warnings.filterwarnings('ignore')

In [16]:
# Load dataframe
df = load_data("../data/movies.csv")
df.head()

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...


## Vue d'ensemble du jeu de données

Vérifier:
- nombre de lignes et de colonnes
- caractéristiques disponibles
- types de données
- valeurs manquantes

In [10]:
#check data types, columns to convert and missing data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9837 entries, 0 to 9836
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Release_Date       9837 non-null   object 
 1   Title              9828 non-null   object 
 2   Overview           9828 non-null   object 
 3   Popularity         9827 non-null   float64
 4   Vote_Count         9827 non-null   object 
 5   Vote_Average       9827 non-null   object 
 6   Original_Language  9827 non-null   object 
 7   Genre              9826 non-null   object 
 8   Poster_Url         9826 non-null   object 
dtypes: float64(1), object(8)
memory usage: 691.8+ KB


In [4]:
#dimensions
df.shape

(9837, 9)

In [6]:
#colomns:features
df.columns.to_list()

['Release_Date',
 'Title',
 'Overview',
 'Popularity',
 'Vote_Count',
 'Vote_Average',
 'Original_Language',
 'Genre',
 'Poster_Url']

## Qualité des données

In [11]:
#Missing data
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

Genre                11
Poster_Url           11
Popularity           10
Vote_Count           10
Vote_Average         10
Original_Language    10
Title                 9
Overview              9
dtype: int64

In [21]:
# duplication
df.duplicated().sum()

np.int64(0)

In [63]:
#statistics
df.describe(include="all")

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
count,9837,9828,9828,9827.000000,9827,9827,9827,9826,9826
unique,5903,9514,9823,NaN,3267,75,44,2337,9826
top,2022-03-10,Beauty and the Beast,Dr. Raichi is one of the only survivors of the...,NaN,0,6.4,en,Drama,https://image.tmdb.org/t/p/original/1g0dhYtq4i...
freq,16,4,2,NaN,100,435,7569,466,1
mean,NaN,NaN,NaN,40.320570,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,108.874308,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,7.100000,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,16.127500,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,21.191000,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,35.174500,NaN,NaN,NaN,NaN,NaN


## Normalisation des types de données

Certaines colonnes nécessitent une normalisation de type :
- Release_Date → datetime
- Release_Year → integer
- Vote_Count → numeric integer
- Vote_Average → numeric numeric integer

Cela garantit que toute valeur inattendue ou mal formée est convertie sans risque en valeur manquante (`NaN`), évitant ainsi de provoquer des erreurs lors de l'analyse et de la visualisation.

In [ ]:
#nettoyage de données
clean_df = preprocess(df)
clean_df.shape

In [ ]:
#changer le type des dates
clean_df["Release_Year"] = (
    clean_df["Release_Year"]
    .astype("Int64")
)

In [ ]:
#Changer le type des valeurs numériques
clean_df["Vote_Count"] = pd.to_numeric(
    clean_df["Vote_Count"],
    errors="coerce"
)

clean_df["Vote_Average"] = pd.to_numeric(
    clean_df["Vote_Average"],
    errors="coerce"
)

#### * Vérification de la colonne Genre

In [22]:
df['Genre'].head(3)

0    Action, Adventure, Science Fiction
1              Crime, Mystery, Thriller
2                              Thriller
Name: Genre, dtype: object

In [24]:
df['Genre'].value_counts()

Genre
Drama                                            466
Comedy                                           403
Drama, Romance                                   248
Horror                                           238
Horror, Thriller                                 199
                                                ... 
Drama, Horror, Thriller, Science Fiction           1
Action, Science Fiction, Animation, Adventure      1
Comedy, Fantasy, Horror, Science Fiction           1
Drama, Science Fiction, Animation                  1
War, Drama, Science Fiction                        1
Name: count, Length: 2337, dtype: int64

La colonne « Genre » contient plusieurs genres séparés par des virgules.

Pour obtenir des statistiques précises sur les genres, il est nécessaire de ventiler cette colonne en genres individuels.

In [36]:
genres_df = clean_df.copy()

genres_df["Genre"] = ( genres_df["Genre"].str.split(","))

In [37]:
genres_df["Genre"].head(3)

0    [Action,  Adventure,  Science Fiction]
1              [Crime,  Mystery,  Thriller]
2                                [Thriller]
Name: Genre, dtype: object

In [38]:
genre_exploded = (
    genres_df
    .explode("Genre")
)

genre_exploded["Genre"] = (
    genre_exploded["Genre"]
    .str.strip()
    .str.title()
)

In [39]:
# obtenir le nombre de genre le plus fréquent (les 20 premiers)
genre_exploded["Genre"].value_counts().head(20)

Genre
Drama              3744
Comedy             3031
Action             2686
Thriller           2488
Adventure          1853
Romance            1476
Horror             1470
Animation          1438
Family             1414
Fantasy            1308
Science Fiction    1273
Crime              1242
Mystery             773
History             427
War                 308
Music               295
Documentary         215
Tv Movie            214
Western             137
Unknown              11
Name: count, dtype: int64

In [40]:
top_genres = (
    genre_exploded["Genre"]
    .value_counts()
    .head(20)
    .reset_index()
)

top_genres.columns = [
    "Genre",
    "Count"
]

In [41]:
fig = px.bar(
    top_genres,
    x="Count",
    y="Genre",
    orientation="h",
    title="Top 20 Genres"
)
fig = px.bar(
    top_genres,
    x="Count",
    y="Genre",
    orientation="h",
    title="Top 20 Movie Genres",
    color="Count",
    color_continuous_scale="Viridis"
)


fig.update_layout(
    template="plotly_white",
    title_font_size=22,
    xaxis_title="Number of movies",
    yaxis_title="Genre",
    height=700,
    margin=dict(l=150)
)
fig.show()


#### * Vérification de la colonne Langues originales.     

In [46]:
clean_df["Original_Language"].value_counts().head(10)

Original_Language
en    7569
ja     645
es     339
fr     292
ko     170
zh     129
it     123
cn     109
ru      83
de      82
Name: count, dtype: int64

In [47]:
#Graphique
language_count = (
    clean_df["Original_Language"]
    .value_counts()
    .head(10)
    .reset_index()
)

language_count.columns = [
    "Language",
    "Count"
]


fig = px.bar(
    language_count,
    x="Language",
    y="Count",
    title="Top Original Languages",
    color="Count",
    template="plotly_white"
)

fig.show()

L'ensemble de données est majoritairement composé de films en anglais.

D'autres langues, comme le japonais et l'espagnol, représentent une part plus réduite du catalogue.

Ceci peut servir d'indicateur sur un tableau de bord pour illustrer la diversité du catalogue.

In [64]:
# Top 5 languages
language_count = (
    clean_df["Original_Language"]
    .value_counts()
)

top5 = language_count.head(5)

# Group remaining languages into "Other"
other_count = language_count.iloc[5:].sum()

language_pie = top5.copy()
language_pie["Other"] = other_count

language_pie = (
    language_pie
    .reset_index()
)

language_pie.columns = [
    "Language",
    "Count"
]

In [65]:
import plotly.express as px

fig = px.pie(
    language_pie,
    names="Language",
    values="Count",
    title="Distribution of Original Languages",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.update_traces(
    textposition="inside",
    textinfo="label+percent"
)

fig.update_layout(
    template="plotly_white",
    title_font_size=20,
    legend_title="Language"
)

fig.show()

#### * Vérification des dates.     

In [52]:
print(f'Release_Year_min: {clean_df["Release_Year"].min()} \nRelease_Year_max {clean_df["Release_Year"].max()}' )

Release_Year_min: 1902 
Release_Year_max 2024


In [53]:
year_count = (
    clean_df
    .groupby("Release_Year")
    .size()
    .reset_index(name="Count")
)

fig = px.line(
    year_count,
    x="Release_Year",
    y="Count",
    title="Movies released per year",
    template="plotly_white"
)

fig.show()

Beaucoup de catalogues de films ont une distribution plus forte après l'an 2000

#### * Vérification de la popularité.     

In [54]:
clean_df["Popularity"].describe()

count    9827.000000
mean       40.320570
std       108.874308
min         7.100000
25%        16.127500
50%        21.191000
75%        35.174500
max      5083.954000
Name: Popularity, dtype: float64

La distribution de la popularité est fortement asymétrique.
Quelques films présentent des scores de popularité très élevés par rapport à la majeure partie du jeu de données.

Nous utilisons :
- une boîte à moustaches pour identifier les valeurs aberrantes

In [57]:
fig = px.box(
    clean_df,
    y="Popularity",
    title="Popularity outliers",
    template="plotly_white"
)

fig.update_layout(
    height=600,
    yaxis_title="Popularity"
)

fig.show()

Le boxplot montre que la distribution est très asymétrique et qu'il existe des valeurs extrêmes (au-delà de la borne supérieure, upper fence ≈ 63.75). Ces valeurs correspondent à des films exceptionnellement populaires.   
=> Pour certaines visualisations, il sera pertinent de limiter ou filtrer les valeurs extrêmes afin d'améliorer la lisibilité des graphiques, tout en conservant le jeu de données complet pour les analyses

## Analyse de la relation entre popularité et note moyenne

L'objectif de cette visualisation est d'étudier la relation entre la popularité d'un film et sa note moyenne.

Afin d'améliorer la lisibilité du graphique :

- seuls les films ayant obtenu au moins 10 votes sont conservés ;
- les 1 % des valeurs de popularité les plus élevées sont exclus de cette visualisation uniquement ;
- le premier genre est utilisé comme genre principal du film.

Ces choix permettent d'obtenir une représentation plus lisible sans modifier le jeu de données d'origine.

* Préparation des données

In [66]:
# créer une copie pour le scatter plot
scatter_df = clean_df.copy()

# conserver seulement les films qui ont un nombre suffisant de votes.
scatter_df = scatter_df[
    scatter_df["Vote_Count"] >= 10
].copy()

# conserver uniquement que le genre principal.
scatter_df["Primary_Genre"] = (
    scatter_df["Genre"]
    .str.split(",")
    .str[0]
    .str.strip()
)

# supprimer le 1 % supérieur des valeurs de popularité à des fins de visualisation.
popularity_threshold = scatter_df["Popularity"].quantile(0.99)

scatter_df = scatter_df[
    scatter_df["Popularity"] <= popularity_threshold
]

* Visualisation

In [68]:
fig = px.scatter(
    scatter_df,
    x="Popularity",
    y="Vote_Average",
    color="Primary_Genre",
    hover_name="Title",
    hover_data={
        "Vote_Count": True,
        "Release_Year": True,
        "Popularity": ":.1f",
        "Vote_Average": ":.1f"
    },
    title="Popularity vs Vote Average by Primary Genre",
    template="plotly_white",
    opacity=0.75
)

fig.update_layout(
    height=650,
    xaxis_title="Popularity",
    yaxis_title="Average Rating",
    legend_title="Primary Genre"
)

fig.show()

- Les points sont très dispersés.   
- La majorité des films se concentre dans une plage de popularité faible à moyenne.     
- Les œuvres les plus populaires ne sont pas systématiquement les mieux notées.   
-  La majorité des points se situe entre 0 et 100 de popularité.Les oeuvres du jeu de données ont une popularité relativement faible.   
- La majorité des notes entre 6 et 8.   
- Les différentes couleurs sont largement mélangées. Ceci suggère que le genre seul n'explique ni la popularité ni la note moyenne.


## Décisions relatives au prétraitement des données

Suite à l'analyse exploratoire, les transformations suivantes seront intégrées dans le pipeline de préparation des données :

- Conversion de `Release_Date` en format datetime.
- Création de la variable `Release_Year`.
- Conversion des variables numériques :
  - `Vote_Count` en entier ;
  - `Vote_Average` en valeur numérique.
- Gestion des valeurs manquantes :
  - remplacement des descriptions absentes ;
  - gestion des genres manquants.
- Normalisation de la colonne `Genre` :
  - séparation des genres multiples ;
  - nettoyage des espaces ;
  - uniformisation du format.
- Conservation des valeurs extrêmes dans le dataset original.
  Les filtres sur la popularité seront appliqués uniquement pour certaines visualisations afin d'améliorer la lisibilité.